# Trident Processing Example
## Segmentation $\rightarrow$ Patching $\rightarrow$ Feature Extraction

The Trident Python package offers a convenient interface for processing whole slide images (WSI) including: tissue vs. background segmentation, tissue patching, and feature extraction ([A. Zhang et al. 2025](https://arxiv.org/abs/2502.06750)).

Example follows tutorials notebook by Trident ([Link](https://github.com/mahmoodlab/TRIDENT/tree/main/tutorials)).

## Imports | Paths | Constants | Device

In [ ]:
import os
import sys
import ast
import h5py
import torch
import openslide
import pandas as pd
from PIL import Image
import geopandas as gpd
from pathlib import Path
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
TRIDENT_ROOT = PROJECT_ROOT / "TRIDENT"
if not TRIDENT_ROOT.exists():
    raise FileNotFoundError(f"Could not find local TRIDENT directory at: {TRIDENT_ROOT}")
if str(TRIDENT_ROOT) not in sys.path:
    sys.path.insert(0, str(TRIDENT_ROOT))

from trident import OpenSlideWSI
from trident.patch_encoder_models import encoder_factory
from trident.segmentation_models import segmentation_model_factory

In [ ]:
ALL_SLIDES_PATH = "../data/raw/slides/pilot100_slides"
SURVIVAL_TABLE_PATH = "../data/interim/matched_clinical_pilot100_luad.csv"

OUT_PATH = "../data/processed/tutorial-1/"
os.makedirs(OUT_PATH, exist_ok=True)

In [ ]:
TARGET_MAG = 20
PATCH_SIZE = 256

PATCH_ENCODER = "uni_v1"

In [ ]:
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

# Select Slide from Survival Table
---

In [ ]:
survival_table = pd.read_csv(SURVIVAL_TABLE_PATH)

file_id_list = ast.literal_eval(survival_table["file_ids"].iloc[0])
file_id = file_id_list[0]
wsi_file_name_list = ast.literal_eval(survival_table["file_names"].iloc[0])
wsi_file_name = wsi_file_name_list[0]

SINGLE_SLIDE_PATH = f"../data/raw/slides/pilot100_slides/{file_id}/{wsi_file_name}"

print(f"File ID: {file_id}")
print(f"WSI Filename: {wsi_file_name}")
print(f"Single Slide Path: {SINGLE_SLIDE_PATH}")

## Create OpenSlideWSI

In [ ]:
slide = OpenSlideWSI(slide_path=SINGLE_SLIDE_PATH, lazy_init=False)

os_slide = openslide.OpenSlide(SINGLE_SLIDE_PATH)

## Display Slide

In [ ]:
thumbnail = os_slide.get_thumbnail((1000, 1000))
display(thumbnail)

# Run Segmentation
---

In [ ]:
segmentation_model = segmentation_model_factory("hest")

geojson_contours = slide.segment_tissue(
    segmentation_model=segmentation_model,
    target_mag=TARGET_MAG,
    job_dir=OUT_PATH,
    device="mps",
    num_workers=0,
    batch_size=1
)

## Visualize Contours

In [ ]:
contour_image = Image.open(os.path.join(OUT_PATH, 'contours', wsi_file_name.replace('.svs', '.jpg')))
display(contour_image)

##  Check Contours Saved into GeoJSON with GeoPandas

In [ ]:
gdf = gpd.read_file(geojson_contours)
gdf.head(n=10)

# Tissue Coordinate Extraction
---

## Run Patch Coordinate Extraction

In [ ]:
coords_path = slide.extract_tissue_coords(
    target_mag=TARGET_MAG,
    patch_size=PATCH_SIZE,
    save_coords=OUT_PATH
)

## Visualize

In [ ]:
viz_coords_path = slide.visualize_coords(
    coords_path=coords_path,
    save_patch_viz=os.path.join(OUT_PATH, "visualization")
)
display(Image.open(viz_coords_path))

## Inspect h5 with Patch Coordinates

In [ ]:
def print_attrs(name, obj):
    print(f"Object: {name}")
    for key, value in obj.attrs.items():
        print(f"  Attribute - {key}: {value}")

with h5py.File(coords_path, 'r') as h5_file:
    print("Contents and Attributes in patch coords file:")
    h5_file.visititems(print_attrs)

# Patch Feature Extraction with the UNI Model
---

## Instantiate UNI Model using the Factory 

In [ ]:
encoder = encoder_factory(PATCH_ENCODER)
encoder.eval()
encoder.to(DEVICE)

## Run UNI Feature Extraction

In [ ]:
features_dir = os.path.join(OUT_PATH, f"features_{PATCH_ENCODER}")
feats_path = slide.extract_patch_features(
    patch_encoder=encoder,
    coords_path=coords_path,
    save_features=features_dir,
    device=DEVICE
)

## Inspect h5 with Patch Features 

In [ ]:
with h5py.File(feats_path, 'r') as h5_file:
    print("Contents and Attributes in feats file:")
    h5_file.visititems(print_attrs)